In [1]:
import os
from dotenv import load_dotenv
from typing import TypedDict, List
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.chat_models import AzureChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage

In [2]:
load_dotenv()
tavily_tool = TavilySearchResults(max_results=3)
tools = [tavily_tool]


In [3]:
llm = AzureChatOpenAI(
    deployment_name=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    openai_api_version=os.environ["OPENAI_API_VERSION"],
    openai_api_key=os.environ["AZURE_OPENAI_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"]
)


C:\Users\rishi\AppData\Local\Temp\ipykernel_14108\3388569776.py:1: LangChainDeprecationWarning: The class `AzureChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import AzureChatOpenAI``.
  llm = AzureChatOpenAI(


In [4]:
# Define AgentState holding the conversation messages
class AgentState(TypedDict):
    messages: List[BaseMessage]

In [5]:
# Agent node: sends current messages to LLM and appends the response
def run_agent(state: AgentState) -> AgentState:
    print("Running agent with messages:")
    for m in state["messages"]:
        print(f"- {m.content}")
    response = llm.invoke(state["messages"])
    print("Agent response:", response.content)
    return {"messages": state["messages"] + [response]}

In [6]:
tool_node = ToolNode(tools)

In [7]:
# Function to route execution based on presence of tool calls in messages
def check_tool_calls(state: AgentState) -> str:
    for msg in state["messages"]:
        if hasattr(msg, "additional_kwargs") and "tool_calls" in msg.additional_kwargs:
            print("Tool calls detected, routing to tools node")
            return "tools"
    print("No tool calls detected, ending")
    return END

In [8]:
builder = StateGraph(state_schema=AgentState)
builder.add_node("agent", RunnableLambda(run_agent))
builder.add_node("tools", tool_node)
builder.set_entry_point("agent")
builder.add_conditional_edges("agent", check_tool_calls, {
    "tools": "tools",
    END: END,
})

In [9]:
# After tools run, go back to agent to process the tool outputs
builder.add_edge("tools", "agent")

In [10]:
graph = builder.compile()

In [11]:
# Run the graph with initial user input
initial_messages = [HumanMessage(content="What's the latest news about SpaceX?")]

In [12]:
for step in graph.stream({"messages": initial_messages}):
    msgs = step.get("messages", [])
    print("\nConversation so far:")
    for m in msgs:
        print(f"- {m.content}")

Running agent with messages:
- What's the latest news about SpaceX?
Agent response: I don't have access to real-time news updates, but as of my last knowledge update in October 2023, SpaceX was actively involved in several key initiatives, including ongoing launches of its Starlink satellites, development of the Starship spacecraft for missions to the Moon and Mars, and collaborations with NASA for various projects. They were also working on increasing the frequency of their Falcon 9 launches and expanding their commercial launch services.

For the latest news, I recommend checking SpaceX's official website or reputable news sources that cover space exploration and technology.
No tool calls detected, ending

Conversation so far:


In [13]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List
import operator

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.chat_models import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.runnables import RunnableLambda
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

load_dotenv()

# Prepare the Tavily tool with API key
tavily_tool = TavilySearchResults(api_key=os.environ["TAVILY_API_KEY"], max_results=3)
tools = [tavily_tool]

# Create Azure OpenAI LLM
llm = AzureChatOpenAI(
    deployment_name=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    openai_api_version=os.environ["OPENAI_API_VERSION"],
    openai_api_key=os.environ["AZURE_OPENAI_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    temperature=0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

# System message describing the tool and instructing usage
system_message = SystemMessage(
    content=(
        "You are a helpful assistant with access to a web search tool named 'TavilySearchResults'.\n"
        "Use this tool to get the latest, real-time information when asked about recent events or news.\n"
        "When you don't know an answer or the question involves current info, call the tool."
    )
)

# Define prompt template including system message and placeholders for messages and scratchpad
prompt = ChatPromptTemplate.from_messages([
    system_message,
    MessagesPlaceholder(variable_name="messages"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Define LangGraph AgentState schema
class AgentState(TypedDict):
    messages: Annotated[List[HumanMessage | AIMessage | ToolMessage], operator.add]

# Define the agent node function using AzureChatOpenAI LLM and Tavily tool integration
def run_agent(state: AgentState) -> AgentState:
    # Compose the input messages with system message + conversation so far
    input_messages = [system_message] + state["messages"]

    # Invoke LLM with streaming enabled and callback for printing
    response = llm.invoke(input_messages)

    # Append LLM response to messages
    new_messages = state["messages"] + [response]

    # Check if LLM suggested a tool call in additional_kwargs
    tool_calls = getattr(response, "additional_kwargs", {}).get("tool_calls", None)

    # If there's a tool call, execute tool node and add tool response to messages
    if tool_calls:
        tool_node = ToolNode(tools)
        tool_state = {"messages": new_messages}
        tool_result_state = tool_node.invoke(tool_state)
        new_messages += tool_result_state["messages"]

    return {"messages": new_messages}

# Build the LangGraph
builder = StateGraph(state_schema=AgentState)
builder.add_node("agent", RunnableLambda(run_agent))
builder.set_entry_point("agent")
builder.set_finish_point("agent")
graph = builder.compile()

# Initial user message
initial_messages = [HumanMessage(content="What's the latest news about SpaceX?")]

# Run graph and stream outputs
for step in graph.stream({"messages": initial_messages}):
    msgs = step.get("messages", [])
    print("\nConversation so far:")
    for msg in msgs:
        print(f"- {msg.content}")


Let me check the latest news about SpaceX for you. One moment, please.
Conversation so far:
